# Práctica 1: SparkSQL
## Tarea 2 - Heterogeneity Dataset for Human Activity Recognition (HHAR)

En esta tarea se procesan datos procedentes de los sensores de acelerómetro y
giroscopio de teléfonos y relojes mediante DataFrames de Spark.

El objetivo es agrupar los datos utilizando como clave `User`, `Model` y `gt`,
obtener estadísticas de los movimientos registrados y combinar posteriormente
los resultados de los distintos sensores y dispositivos.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Practica1_Tarea2_SparkSQL") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark funcionando correctamente")
print("Versión de Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 13:10:18 WARN Utils: Your hostname, MacBook-Air-de-Carmen-2.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
26/11/11 13:10:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 13:10:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark funcionando correctamente
Versión de Spark: 4.2.0


In [2]:
# Comprobamos la cabecera y las primeras filas del fichero

with open("Phones_accelerometer.csv", "r") as fichero:
    for i in range(5):
        print(fichero.readline().strip())

0,1424696633908,1424696631913248572,-5.958191,0.6880646,8.135345,a,nexus4,nexus4_1,stand
1,1424696633909,1424696631918283972,-5.95224,0.6702118,8.136536,a,nexus4,nexus4_1,stand
2,1424696633918,1424696631923288855,-5.9950867,0.6535491999999999,8.204376,a,nexus4,nexus4_1,stand
3,1424696633919,1424696631928385290,-5.9427185,0.6761626999999999,8.128204,a,nexus4,nexus4_1,stand
4,1424696633929,1424696631933420691,-5.991516000000001,0.64164734,8.135345,a,nexus4,nexus4_1,stand


## 1. Definición del esquema

Los cuatro ficheros CSV del dataset HHAR tienen la misma estructura.
Se define explícitamente el esquema que se utilizará para crear los DataFrames,
sin utilizar inferencia automática de tipos.

In [3]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType
)

# Esquema de los ficheros HHAR
schema = StructType([
    StructField("Index", LongType(), True),
    StructField("Arrival_Time", LongType(), True),
    StructField("Creation_Time", LongType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("z", DoubleType(), True),
    StructField("User", StringType(), True),
    StructField("Model", StringType(), True),
    StructField("Device", StringType(), True),
    StructField("gt", StringType(), True)
])

print(schema)

StructType([StructField('Index', LongType(), True), StructField('Arrival_Time', LongType(), True), StructField('Creation_Time', LongType(), True), StructField('x', DoubleType(), True), StructField('y', DoubleType(), True), StructField('z', DoubleType(), True), StructField('User', StringType(), True), StructField('Model', StringType(), True), StructField('Device', StringType(), True), StructField('gt', StringType(), True)])


## 2. Creación de los DataFrames

Se define una función para cargar los ficheros CSV utilizando el esquema
definido anteriormente. A continuación se crean los cuatro DataFrames
correspondientes al acelerómetro y giroscopio de teléfonos y relojes.

In [4]:
def cargar_dataframe(ruta):
    """
    Carga un fichero CSV del dataset HHAR en un DataFrame de Spark.

    Parámetros:
        ruta (str): Ruta del fichero CSV que se desea cargar.

    Retorna:
        DataFrame: DataFrame de Spark utilizando el esquema HHAR definido.
    """
    return spark.read.csv(
        ruta,
        schema=schema,
        header=False
    )

In [5]:
# Creamos un DataFrame para cada fichero

phones_acc = cargar_dataframe("Phones_accelerometer.csv")
phones_gyro = cargar_dataframe("Phones_gyroscope.csv")

watch_acc = cargar_dataframe("Watch_accelerometer.csv")
watch_gyro = cargar_dataframe("Watch_gyroscope.csv")

print("DataFrames creados correctamente")

DataFrames creados correctamente


In [6]:
# Comprobamos el esquema del DataFrame
phones_acc.printSchema()

root
 |-- Index: long (nullable = true)
 |-- Arrival_Time: long (nullable = true)
 |-- Creation_Time: long (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)



In [7]:
# Mostramos las primeras 5 filas
phones_acc.show(5)

+-----+-------------+-------------------+------------------+------------------+--------+----+------+--------+-----+
|Index| Arrival_Time|      Creation_Time|                 x|                 y|       z|User| Model|  Device|   gt|
+-----+-------------+-------------------+------------------+------------------+--------+----+------+--------+-----+
|    0|1424696633908|1424696631913248572|         -5.958191|         0.6880646|8.135345|   a|nexus4|nexus4_1|stand|
|    1|1424696633909|1424696631918283972|          -5.95224|         0.6702118|8.136536|   a|nexus4|nexus4_1|stand|
|    2|1424696633918|1424696631923288855|        -5.9950867|0.6535491999999999|8.204376|   a|nexus4|nexus4_1|stand|
|    3|1424696633919|1424696631928385290|        -5.9427185|0.6761626999999999|8.128204|   a|nexus4|nexus4_1|stand|
|    4|1424696633929|1424696631933420691|-5.991516000000001|        0.64164734|8.135345|   a|nexus4|nexus4_1|stand|
+-----+-------------+-------------------+------------------+------------

## 3. Agregación de los movimientos

Los datos se agrupan utilizando como clave primaria la terna formada por
`User`, `Model` y `gt`.

Para cada grupo se calculan la media, la desviación estándar, el valor máximo
y el valor mínimo de los tres ejes de movimiento (`x`, `y` y `z`).

In [8]:
from pyspark.sql import functions as F

def agregar_movimientos(df, prefijo):
    """
    Agrupa los datos de movimiento por usuario, modelo y actividad.

    Para cada grupo calcula la media, desviación estándar,
    valor máximo y valor mínimo de los ejes x, y y z.

    Parámetros:
        df (DataFrame): DataFrame de Spark con los datos del sensor.
        prefijo (str): Prefijo utilizado para identificar las columnas
                       resultantes (acc o gyro).

    Retorna:
        DataFrame: DataFrame agregado por User, Model y gt.
    """
    
    return df.groupBy("User", "Model", "gt").agg(
        
        F.mean("x").alias(f"{prefijo}_x_media"),
        F.stddev("x").alias(f"{prefijo}_x_std"),
        F.max("x").alias(f"{prefijo}_x_max"),
        F.min("x").alias(f"{prefijo}_x_min"),
        
        F.mean("y").alias(f"{prefijo}_y_media"),
        F.stddev("y").alias(f"{prefijo}_y_std"),
        F.max("y").alias(f"{prefijo}_y_max"),
        F.min("y").alias(f"{prefijo}_y_min"),
        
        F.mean("z").alias(f"{prefijo}_z_media"),
        F.stddev("z").alias(f"{prefijo}_z_std"),
        F.max("z").alias(f"{prefijo}_z_max"),
        F.min("z").alias(f"{prefijo}_z_min")
    )

In [9]:
# Agregamos los cuatro DataFrames

phones_acc_agg = agregar_movimientos(phones_acc, "acc")
phones_gyro_agg = agregar_movimientos(phones_gyro, "gyro")

watch_acc_agg = agregar_movimientos(watch_acc, "acc")
watch_gyro_agg = agregar_movimientos(watch_gyro, "gyro")

print("DataFrames agregados correctamente")

DataFrames agregados correctamente


## 4. Unión de acelerómetro y giroscopio

Una vez agregados los datos, se realiza un `join` utilizando como claves
`User`, `Model` y `gt`.

Se realiza una unión entre acelerómetro y giroscopio para los teléfonos y
otra unión entre acelerómetro y giroscopio para los relojes.

In [10]:
# Unimos acelerómetro y giroscopio de los teléfonos
phones_join = phones_acc_agg.join(
    phones_gyro_agg,
    on=["User", "Model", "gt"],
    how="inner"
)

# Unimos acelerómetro y giroscopio de los relojes
watch_join = watch_acc_agg.join(
    watch_gyro_agg,
    on=["User", "Model", "gt"],
    how="inner"
)

print("Joins realizados correctamente")

Joins realizados correctamente


## 5. Creación del DataFrame final

Finalmente, se unen los resultados correspondientes a teléfonos y relojes
mediante una operación `union`, obteniendo un único DataFrame con las
estadísticas calculadas para los distintos sensores y movimientos.

In [11]:
# Unimos los DataFrames de teléfonos y relojes
# para obtener el DataFrame final

df_final = phones_join.unionByName(watch_join)

print("DataFrame final creado correctamente")

DataFrame final creado correctamente


## 6. Resultado final

Se comprueba el esquema del DataFrame final y se muestran algunos registros
para verificar que las operaciones de agregación, `join` y `union` se han
realizado correctamente.

In [12]:
# Comprobamos el esquema del DataFrame final
df_final.printSchema()

root
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- gt: string (nullable = true)
 |-- acc_x_media: double (nullable = true)
 |-- acc_x_std: double (nullable = true)
 |-- acc_x_max: double (nullable = true)
 |-- acc_x_min: double (nullable = true)
 |-- acc_y_media: double (nullable = true)
 |-- acc_y_std: double (nullable = true)
 |-- acc_y_max: double (nullable = true)
 |-- acc_y_min: double (nullable = true)
 |-- acc_z_media: double (nullable = true)
 |-- acc_z_std: double (nullable = true)
 |-- acc_z_max: double (nullable = true)
 |-- acc_z_min: double (nullable = true)
 |-- gyro_x_media: double (nullable = true)
 |-- gyro_x_std: double (nullable = true)
 |-- gyro_x_max: double (nullable = true)
 |-- gyro_x_min: double (nullable = true)
 |-- gyro_y_media: double (nullable = true)
 |-- gyro_y_std: double (nullable = true)
 |-- gyro_y_max: double (nullable = true)
 |-- gyro_y_min: double (nullable = true)
 |-- gyro_z_media: double (nullable = true)
 |-- g

In [13]:
# Mostramos algunos registros del DataFrame final
df_final.show(10, truncate=False)

26/11/11 13:44:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 2:====================================================>     (9 + 1) / 10]

+----+------+----------+-------------------+------------------+-------------------+-------------------+--------------------+--------------------+-----------------+-------------------+------------------+-------------------+------------------+-----------------+---------------------+--------------------+------------------+-------------------+----------------------+--------------------+------------------+-------------------+---------------------+--------------------+------------------+-------------------+
|User|Model |gt        |acc_x_media        |acc_x_std         |acc_x_max          |acc_x_min          |acc_y_media         |acc_y_std           |acc_y_max        |acc_y_min          |acc_z_media       |acc_z_std          |acc_z_max         |acc_z_min        |gyro_x_media         |gyro_x_std          |gyro_x_max        |gyro_x_min         |gyro_y_media          |gyro_y_std          |gyro_y_max        |gyro_y_min         |gyro_z_media         |gyro_z_std          |gyro_z_max        |gyro_z_